# Chapter 08 — Simple Linear Regression

In this notebook we build understanding from the ground up: what is the regression
problem, how does Ordinary Least Squares work, and how do we implement, interpret,
and diagnose a simple linear regression model in Python.

---
## 1. The Regression Problem

In **supervised learning** we have two broad families of problems:

| Task | Target type | Example |
|------|------------|---------|
| **Classification** | Discrete / categorical | Spam vs. not spam |
| **Regression** | Continuous / numerical | Predicting house prices |

Regression asks: *given one or more input features, can we predict a continuous
numerical output?*

Linear regression is the simplest and most interpretable regression model.
It assumes the relationship between the input feature $x$ and the target $y$
is approximately **linear**.

---
## 2. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## 3. Generating Synthetic Data

We create a dataset where the true relationship is:

$$y = 3x + 7 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

This way we **know** the ground-truth parameters and can verify that our model
recovers them.

In [ ]:
rng = np.random.default_rng(42)

n_samples = 100
TRUE_SLOPE = 3.0
TRUE_INTERCEPT = 7.0
NOISE_STD = 4.0

x = rng.uniform(0, 10, size=n_samples)
noise = rng.normal(0, NOISE_STD, size=n_samples)
y = TRUE_SLOPE * x + TRUE_INTERCEPT + noise

print(f'x shape: {x.shape}')
print(f'y shape: {y.shape}')
print(f'First 5 x values: {x[:5].round(2)}')
print(f'First 5 y values: {y[:5].round(2)}')

In [ ]:
plt.scatter(x, y, alpha=0.6, edgecolors='k', linewidths=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Data — Is There a Linear Trend?')
plt.show()

The scatter plot reveals a clear upward trend — a line should fit this well.

---
## 4. The Simple Linear Regression Model

The model assumes:

$$\hat{y} = \beta_0 + \beta_1 x$$

where:

- $\beta_0$ is the **intercept** (the predicted value when $x = 0$)
- $\beta_1$ is the **slope** (how much $\hat{y}$ changes for a one-unit increase in $x$)
- $\hat{y}$ is the **predicted** value (as opposed to the observed $y$)

The **residual** for the $i$-th data point is the prediction error:

$$e_i = y_i - \hat{y}_i$$

---
## 5. Ordinary Least Squares (OLS)

### 5.1 The Cost Function

OLS finds $\beta_0$ and $\beta_1$ by **minimising the sum of squared residuals**
(also called the Residual Sum of Squares, or RSS):

$$\text{RSS} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \sum_{i=1}^{n} (y_i - \beta_0 - \beta_1 x_i)^2$$

### 5.2 Why Squared Errors?

- **Absolute errors** ($|e_i|$) are not differentiable at zero, making optimisation harder.
- **Squared errors** ($e_i^2$) are smooth and convex — there is a single global minimum.
- Squaring penalises large errors more heavily, pushing the model to avoid big mistakes.

### 5.3 Closed-Form Solution

Taking derivatives of RSS with respect to $\beta_0$ and $\beta_1$, setting them to zero,
and solving gives the **normal equations**:

$$\beta_1 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

$$\beta_0 = \bar{y} - \beta_1 \bar{x}$$

where $\bar{x}$ and $\bar{y}$ are the sample means.

### 5.4 Manual Computation

Let's compute the OLS estimates by hand to build intuition before using sklearn.

In [ ]:
x_mean = x.mean()
y_mean = y.mean()

beta_1_manual = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
beta_0_manual = y_mean - beta_1_manual * x_mean

print(f'Manual OLS estimates:')
print(f'  Slope     (beta_1): {beta_1_manual:.4f}  (true: {TRUE_SLOPE})')
print(f'  Intercept (beta_0): {beta_0_manual:.4f}  (true: {TRUE_INTERCEPT})')

The estimates are close to the true values. The small discrepancy comes from the
random noise — with more data or less noise they would converge to the true parameters.

### 5.5 Visualising the Cost Surface

The RSS is a **paraboloid** — a bowl-shaped surface in ($\beta_0$, $\beta_1$) space.
OLS finds the bottom of the bowl.

In [ ]:
# Create a grid of (beta_0, beta_1) values
b0_range = np.linspace(beta_0_manual - 10, beta_0_manual + 10, 80)
b1_range = np.linspace(beta_1_manual - 3, beta_1_manual + 3, 80)
B0, B1 = np.meshgrid(b0_range, b1_range)

# Compute RSS for every combination
RSS = np.zeros_like(B0)
for i in range(len(b0_range)):
    for j in range(len(b1_range)):
        predictions = B0[j, i] + B1[j, i] * x
        RSS[j, i] = np.sum((y - predictions) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot
cs = axes[0].contour(B0, B1, RSS, levels=30, cmap='viridis')
axes[0].clabel(cs, inline=True, fontsize=7)
axes[0].plot(beta_0_manual, beta_1_manual, 'r*', markersize=15, label='OLS solution')
axes[0].set_xlabel(r'$\beta_0$ (intercept)')
axes[0].set_ylabel(r'$\beta_1$ (slope)')
axes[0].set_title('RSS Contour Plot')
axes[0].legend()

# 3-D surface
ax3d = fig.add_subplot(122, projection='3d')
ax3d.plot_surface(B0, B1, RSS, cmap='viridis', alpha=0.8, edgecolor='none')
ax3d.set_xlabel(r'$\beta_0$')
ax3d.set_ylabel(r'$\beta_1$')
ax3d.set_zlabel('RSS')
ax3d.set_title('RSS Surface (the "bowl")')
# Remove the flat contour axes since we replaced it with 3D
axes[1].remove()

plt.tight_layout()
plt.show()

The red star marks the OLS solution — the point at the very bottom of the bowl.
Because the surface is convex (bowl-shaped), any optimisation algorithm will find
the same unique minimum.

---
## 6. Implementation with scikit-learn

### 6.1 Preparing the Data

sklearn expects the feature matrix `X` to be 2-D (samples, features).
Our `x` is 1-D, so we reshape it.

In [ ]:
X = x.reshape(-1, 1)  # shape (100, 1)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

### 6.2 Train / Test Split

We hold out 20% of the data to evaluate the model on unseen samples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

### 6.3 Fitting the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f'sklearn estimates:')
print(f'  Slope     (coef_):      {model.coef_[0]:.4f}')
print(f'  Intercept (intercept_): {model.intercept_:.4f}')

Note: the sklearn estimates differ slightly from our manual computation because
the manual calculation used all 100 points, while sklearn was fitted on the 80
training samples.

---
## 7. Interpreting the Coefficients

- **Intercept** ($\beta_0$): the predicted value of $y$ when $x = 0$.  
  In our model this means: when the feature is zero, we predict $y \approx \beta_0$.

- **Slope** ($\beta_1$): for every **one-unit increase** in $x$, the predicted $y$
  increases by $\beta_1$ units.  
  A positive slope means $y$ and $x$ move in the same direction.

In [ ]:
print(f'Interpretation:')
print(f'  When x = 0, predicted y = {model.intercept_:.2f}')
print(f'  For each +1 in x, predicted y increases by {model.coef_[0]:.2f}')

---
## 8. Visualising the Regression Line

In [ ]:
# Generate predictions across the full range of x
x_line = np.linspace(0, 10, 200).reshape(-1, 1)
y_line = model.predict(x_line)

plt.scatter(X_train, y_train, alpha=0.5, label='Train', edgecolors='k', linewidths=0.5)
plt.scatter(X_test, y_test, alpha=0.5, label='Test', marker='s', edgecolors='k', linewidths=0.5)
plt.plot(x_line, y_line, 'r-', linewidth=2, label='Regression line')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Simple Linear Regression — Fitted Line')
plt.legend()
plt.show()

The red line passes through the cloud of points, capturing the linear trend.
Both training (circles) and test (squares) points fall around the line.

---
## 9. Making Predictions and Computing Metrics

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print('Training set:')
print(f'  MSE:  {mean_squared_error(y_train, y_train_pred):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}')
print(f'  R2:   {r2_score(y_train, y_train_pred):.4f}')

print('\nTest set:')
print(f'  MSE:  {mean_squared_error(y_test, y_test_pred):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}')
print(f'  R2:   {r2_score(y_test, y_test_pred):.4f}')

---
## 10. Understanding R² (Coefficient of Determination)

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- $\text{SS}_{\text{res}}$: residual sum of squares — the error left after fitting the model.
- $\text{SS}_{\text{tot}}$: total sum of squares — the variance in $y$ around its mean.

**Interpretation:**
- $R^2 = 1$: the model explains **all** the variance — perfect predictions.
- $R^2 = 0$: the model explains **none** of the variance — predicting $\bar{y}$ every time
  would do just as well.
- $R^2 < 0$: the model is **worse** than simply predicting the mean (possible on the test set).

Think of $R^2$ as the **proportion of variance explained** by the model.

In [ ]:
# Manual R² computation to verify
ss_res = np.sum((y_test - y_test_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2_manual = 1 - ss_res / ss_tot

print(f'Manual R² on test set: {r2_manual:.4f}')
print(f'sklearn R² on test set: {r2_score(y_test, y_test_pred):.4f}')

---
## 11. Residual Analysis

Residuals ($e_i = y_i - \hat{y}_i$) contain diagnostic information about the model.
For a well-specified linear model, residuals should:

1. **Have zero mean** — already guaranteed by OLS.
2. **Be roughly normally distributed.**
3. **Show no pattern** when plotted against predicted values (homoscedasticity).
4. **Be independent** of one another.

### 11.1 Residuals vs Predicted Values

In [ ]:
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs predicted
axes[0].scatter(y_train_pred, residuals_train, alpha=0.6, edgecolors='k', linewidths=0.5)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted (Train)')

# Histogram of residuals
axes[1].hist(residuals_train, bins=15, edgecolor='k', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals (Train)')

plt.tight_layout()
plt.show()

**What to look for:**

- **Left plot**: the points should scatter randomly around the red dashed line (zero).
  A funnel shape would indicate heteroscedasticity; a curve would suggest a non-linear
  relationship that the model is missing.
- **Right plot**: the histogram should look roughly bell-shaped (normal).

### 11.2 Residuals vs Feature Values

In [ ]:
plt.scatter(X_train, residuals_train, alpha=0.6, edgecolors='k', linewidths=0.5)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.xlabel('x')
plt.ylabel('Residuals')
plt.title('Residuals vs Feature (x)')
plt.show()

If there is no pattern, the linear assumption is reasonable for this feature.

### 11.3 Q-Q Plot (Normality Check)

A quick visual test for normality: plot the sorted residuals against the
theoretical quantiles of a normal distribution.  If residuals are normal,
points will lie on the diagonal.

In [ ]:
sorted_residuals = np.sort(residuals_train)
n = len(sorted_residuals)
# Theoretical quantiles from standard normal
from scipy import stats
theoretical_quantiles = stats.norm.ppf(np.arange(1, n + 1) / (n + 1))

plt.scatter(theoretical_quantiles, sorted_residuals, alpha=0.6, edgecolors='k', linewidths=0.5)
# Reference line through Q1 and Q3
q1_idx, q3_idx = n // 4, 3 * n // 4
slope_qq = (sorted_residuals[q3_idx] - sorted_residuals[q1_idx]) / (
    theoretical_quantiles[q3_idx] - theoretical_quantiles[q1_idx]
)
intercept_qq = sorted_residuals[q1_idx] - slope_qq * theoretical_quantiles[q1_idx]
qq_line = slope_qq * theoretical_quantiles + intercept_qq
plt.plot(theoretical_quantiles, qq_line, 'r--', linewidth=1)
plt.xlabel('Theoretical Quantiles')
plt.ylabel('Sample Quantiles (Residuals)')
plt.title('Q-Q Plot of Residuals')
plt.show()

Points falling close to the red dashed line confirm that the residuals are
approximately normally distributed — one of the key assumptions of linear regression.

---
## 12. What Happens When the True Relationship Is Not Linear?

Let's see how a simple linear model behaves when the data is actually quadratic.

In [ ]:
# Generate non-linear data
x_nl = rng.uniform(-3, 3, size=120)
y_nl = 2 * x_nl ** 2 + 0.5 * x_nl + 1 + rng.normal(0, 2, size=120)

X_nl = x_nl.reshape(-1, 1)
model_nl = LinearRegression().fit(X_nl, y_nl)
y_nl_pred = model_nl.predict(X_nl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + line
order = np.argsort(x_nl)
axes[0].scatter(x_nl, y_nl, alpha=0.5, edgecolors='k', linewidths=0.5)
axes[0].plot(x_nl[order], y_nl_pred[order], 'r-', linewidth=2)
axes[0].set_title(f'Linear Fit on Quadratic Data  (R² = {r2_score(y_nl, y_nl_pred):.3f})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Residuals
residuals_nl = y_nl - y_nl_pred
axes[1].scatter(y_nl_pred, residuals_nl, alpha=0.5, edgecolors='k', linewidths=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals — Clear Curved Pattern')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

The residual plot shows a **U-shape** — a clear sign the model is missing
a non-linear pattern. This motivates polynomial regression (covered in Notebook 02).

---
## 13. Effect of Sample Size on Estimates

With more data, the OLS estimates converge to the true parameters.
Let's demonstrate this by fitting the model on increasing sample sizes.

In [ ]:
sample_sizes = [10, 25, 50, 100, 250, 500, 1000, 5000]
slopes = []
intercepts = []

for n in sample_sizes:
    xi = rng.uniform(0, 10, size=n)
    yi = TRUE_SLOPE * xi + TRUE_INTERCEPT + rng.normal(0, NOISE_STD, size=n)
    lr = LinearRegression().fit(xi.reshape(-1, 1), yi)
    slopes.append(lr.coef_[0])
    intercepts.append(lr.intercept_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sample_sizes, slopes, 'bo-')
axes[0].axhline(TRUE_SLOPE, color='red', linestyle='--', label=f'True slope = {TRUE_SLOPE}')
axes[0].set_xlabel('Sample size')
axes[0].set_ylabel('Estimated slope')
axes[0].set_title('Slope Estimate vs Sample Size')
axes[0].set_xscale('log')
axes[0].legend()

axes[1].plot(sample_sizes, intercepts, 'go-')
axes[1].axhline(TRUE_INTERCEPT, color='red', linestyle='--', label=f'True intercept = {TRUE_INTERCEPT}')
axes[1].set_xlabel('Sample size')
axes[1].set_ylabel('Estimated intercept')
axes[1].set_title('Intercept Estimate vs Sample Size')
axes[1].set_xscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

As expected, the estimates become more stable and closer to the true values
as the sample size grows — a property called **consistency**.

---
## 14. Predicting New Values

Once fitted, the model can predict $y$ for any new $x$ value.

In [ ]:
new_x = np.array([[2.5], [5.0], [7.5], [10.0]])
new_y_pred = model.predict(new_x)

for xi, yi in zip(new_x.ravel(), new_y_pred):
    print(f'  x = {xi:.1f}  =>  predicted y = {yi:.2f}')

---
## 15. Summary

| Concept | Key Takeaway |
|---------|-------------|
| Regression problem | Predict a continuous target from features |
| Simple linear regression | $\hat{y} = \beta_0 + \beta_1 x$ |
| OLS | Minimises sum of squared residuals — has a closed-form solution |
| Cost function shape | Convex paraboloid — unique global minimum |
| Slope interpretation | Change in $\hat{y}$ per unit change in $x$ |
| R² | Proportion of variance explained (0 = baseline, 1 = perfect) |
| Residual analysis | Check for patterns, normality, constant variance |
| Non-linear data | Linear model misses curvature — residuals show patterns |

**Next:** In Notebook 02 we extend to multiple features and polynomial regression.